# Stage 1 walkthrough: XBRL facts → Parquet → one defensible number

The half of the system that never uses retrieval. Figures come from SQL over XBRL,
or they do not appear — the LLM never does arithmetic on retrieved text.

This notebook follows Synchrony (SYF) from its raw member in `companyfacts.zip` to a
`query_financials` answer, then shows the three traps that make "what was revenue in
2024?" harder than it looks: **tags**, **restatements**, and **fiscal calendars**.

**Prerequisites:** `uv sync --group notebook`, the `.venv` kernel, `fc fetch-companyfacts`
and `fc ingest-facts` already run. No network. Companion reading: `docs/stage1_structured_layer.md`.

## 0 · Setup

In [ ]:
import os
from pathlib import Path

# .env, config/ and data/ are relative to the repo root, exactly as when running `fc`.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import json
import tempfile
import zipfile
from collections import Counter

import duckdb
import pandas as pd

from filing_copilot.config import get_settings
from filing_copilot.edgar import EdgarClient
from filing_copilot.edgar.endpoints import companyfacts_bulk_url
from filing_copilot.structured import Corpus

pd.set_option("display.width", 160)
settings = get_settings()
corpus = Corpus.load(settings.corpus_path)
company = corpus.by_ticker("SYF")
archive_path = EdgarClient(settings).cache.path_for(companyfacts_bulk_url())
print(f"{len(corpus)} companies in the corpus; archive at {archive_path}")
pd.DataFrame([{"ticker": c.ticker, "cik": c.cik, "name": c.name, "group": c.group} for c in corpus])

## 1 · One company's raw facts — straight out of the zip

`zipfile` opens one member without extracting anything. The JSON is four levels deep:

```
facts → taxonomy → tag → units → unit → [observation, ...]
```

In [ ]:
from filing_copilot.structured.ingest import member_name

with zipfile.ZipFile(archive_path) as archive:
    payload = archive.read(member_name(company.cik))

facts = json.loads(payload)
print(f"{len(payload) / 2**20:.1f} MiB of JSON for {facts['entityName']}")
print("top-level keys:", list(facts))
for taxonomy, tags in facts["facts"].items():
    print(f"  {taxonomy:8} {len(tags):>5} tags")

## 2 · Two shapes of fact: durations and instants

An income-statement figure covers a **span** (`start` → `end`). A balance-sheet figure is
measured at an **instant** — it has no `start`. The same `fy`/`fp` labels appear on both,
and they describe the **report that carried the number**, not the number's own period.

In [ ]:
def observations(tag: str) -> pd.DataFrame:
    node = facts["facts"]["us-gaap"][tag]
    unit, obs = next(iter(node["units"].items()))
    print(f"{tag}: {node['label']!r}, unit {unit}, {len(obs)} observations")
    return pd.DataFrame(obs)

display(observations("NetIncomeLoss").tail(4))      # duration: has start
display(observations("Assets").tail(4))             # instant: no start

## 3 · Flatten — one wide, boring row per observation

`iter_fact_rows` walks the nesting and emits `FactRow`s. Once it is flat, every hard
question downstream becomes a SQL filter.

In [ ]:
from filing_copilot.structured.flatten import iter_fact_rows

rows = list(iter_fact_rows(payload, cik=company.cik))
flat = pd.DataFrame(rows)
print(f"{len(flat):,} rows, {flat['tag'].nunique():,} tags, {flat['unit'].nunique()} units")
print("period_type:", flat["period_type"].value_counts().to_dict())
flat.sample(5, random_state=0)

## 4 · Trap: `fy` is the report's year, not the fact's

Comparatives get re-reported in later filings, carrying the *later* report's `fy`.
Partitioning on `fy` would scatter one period across partitions — so the project derives
`period_year` from the fact's own dates and keeps `fy` as an ordinary column.

In [ ]:
mismatch = flat[flat["fy"].notna() & (flat["fy"] != flat["period_year"])]
print(f"{len(mismatch):,} of {len(flat):,} rows have fy != period_year ({len(mismatch) / len(flat):.0%})")
flat.sort_values("end").head(3)[["tag", "start", "end", "period_year", "fy", "fp", "form", "filed"]]

## 5 · `period_year` — the calendar year a fact belongs to

An instant belongs to its own year. A duration belongs to the calendar year holding
**most** of its span (ADR-0004), so Visa's October–September fiscal year is labelled by
the year it mostly falls in, rather than split in two.

In [ ]:
from datetime import date

from filing_copilot.structured.flatten import period_year

examples = {
    "instant, Dec 31":               (None, date(2024, 12, 31)),
    "calendar year":                 (date(2024, 1, 1), date(2024, 12, 31)),
    "Visa FY2024 (Oct-Sep)":         (date(2023, 10, 1), date(2024, 9, 30)),
    "Apple FY2024 (52/53-week)":     (date(2023, 10, 1), date(2024, 9, 28)),
    "a year ending in February":     (date(2023, 3, 1), date(2024, 2, 29)),
}
for name, (start, end) in examples.items():
    print(f"{name:28} {start} -> {end}   period_year = {period_year(start, end)}")

## 6 · Units are free-form — so `unit` is never optional

A query that forgets to filter on unit can add dollars to lawsuit counts.

In [ ]:
print(flat["unit"].value_counts().to_dict())
flat[~flat["unit"].isin(["USD", "shares", "pure", "USD/shares"])][["tag", "unit", "val", "end"]].head(5)

## 7 · Partitioned Parquet — written to a temp directory here

`ingest_company` is what `fc ingest-facts` runs per company. Running it into a temporary
directory shows the layout without touching `data/`. Partitions are
`cik=…/period_year=…`, and the schema is declared, never inferred.

In [ ]:
from filing_copilot.structured.ingest import FACT_SCHEMA, ingest_company

with tempfile.TemporaryDirectory() as tmp, zipfile.ZipFile(archive_path) as archive:
    result = ingest_company(archive, company, Path(tmp))
    parts = sorted(p.relative_to(tmp) for p in Path(tmp).rglob("*.parquet"))

print(result)
print(f"{len(parts)} files, e.g.:")
for p in parts[-3:]:
    print("  ", p)
print()
print(FACT_SCHEMA)

## 8 · Plain SQL over the real fact table

DuckDB reads the Parquet tree directly — no database server, no load step. This is the
same `read_parquet(..., hive_partitioning = true)` the `DuckDBBackend` uses, and Athena
will run the equivalent over S3 in Stage 8.

Notice one period (Synchrony's FY2023 net income) reported **several times** — once in
its own 10-K, again as comparatives in later filings. That repetition is what restatement
handling works with.

In [ ]:
facts_glob = str(settings.facts_dir / "**" / "*.parquet")
con = duckdb.connect()

print(con.execute(f"""
    SELECT count(*) AS rows, count(DISTINCT tag) AS tags, count(DISTINCT unit) AS units,
           count(DISTINCT cik) AS companies
    FROM read_parquet('{facts_glob}', hive_partitioning = true)
""").df())

con.execute(f"""
    SELECT tag, start, "end", val, form, filed, accn
    FROM read_parquet('{facts_glob}', hive_partitioning = true)
    WHERE cik = ? AND tag = 'NetIncomeLoss' AND unit = 'USD'
      AND period_year = 2023 AND datediff('day', start, "end") BETWEEN 330 AND 400
    ORDER BY filed
""", [company.cik]).df()

## 9 · "Total revenue" is not a tag — the concept registry

`concepts.yaml` maps each business concept to an **ordered allow-list** of tags. The
resolver takes the first one each company actually reports, and **records which one
answered**. Five different tags are needed to express "total revenue" across 20 companies.

Read the `rank` column. Rank 0 is the preferred tag; anything higher is a fallback the
answer will disclose. Synchrony resolves at **rank 4, `InterestIncomeOperating`** — gross
interest income, the weakest proxy on the list. The resolver does not hide that; an
analyst comparing SYF's "revenue" to AXP's needs to know it.

In [ ]:
from filing_copilot.structured import ConceptRegistry, DuckDBBackend
from filing_copilot.structured.resolver import DEFAULT_CONCEPTS_PATH, resolve_concept

registry = ConceptRegistry.load(DEFAULT_CONCEPTS_PATH)
concept = registry["total_revenue"]
print(concept.label, "| unit", concept.unit, "|", concept.period_type)
for rank, tag in enumerate(concept.candidates):
    print(f"  rank {rank}: {tag}")

backend = DuckDBBackend(settings.facts_dir)
resolutions = resolve_concept(concept, backend, corpus.ciks, years=(2024, 2024))
by_cik = {c.cik: c.ticker for c in corpus}
table = pd.DataFrame(
    [{"ticker": by_cik[cik], "tag": r.tag, "rank": r.rank} for cik, r in resolutions.items()]
)
print("\ntags used:", table["tag"].value_counts().to_dict())
table

## 10 · Trap: restatements — one period, more than one number

Bank of America's FY2023 `Revenues` was reported in its FY2023 10-K, then **restated** in a
later filing. Both are real. `as_reported` takes the earliest filing (what they said then);
`as_restated`, the default, takes the latest (what they say now). The answer always says
which basis it used, and flags `restated` when they differ (ADR-0003).

In [ ]:
from filing_copilot.structured import AS_REPORTED, AS_RESTATED, query_financials

bac = corpus.by_ticker("BAC")
display(con.execute(f"""
    SELECT tag, "end", val / 1e9 AS usd_bn, form, filed, accn
    FROM read_parquet('{facts_glob}', hive_partitioning = true)
    WHERE cik = ? AND tag = 'Revenues' AND unit = 'USD' AND period_year = 2023
      AND datediff('day', start, "end") BETWEEN 330 AND 400
    ORDER BY filed
""", [bac.cik]).df())

for basis in (AS_REPORTED, AS_RESTATED):
    result = query_financials(registry, backend, corpus, concept_name="total_revenue",
                              companies=(bac,), period_year=2023, basis=basis)
    v = result.values[0]
    print(f"{basis:12} {v.val / 1e9:8.3f} bn   tag={v.tag}  restated={v.restated}  filed={v.filed}")

## 11 · Trap: "FY2024" is not one period

Visa's fiscal 2024 ended in September; Mastercard's in December. `query_financials`
answers both — and warns, because comparing them directly is comparing different quarters
of the economy. It also warns when the two answers came from different tags.

`fiscal_year_end` is read from each company's own annual facts, never assumed:
Apple's 52/53-week year ends on 2024-09-28, not 09-30.

In [ ]:
visa, mastercard = corpus.by_ticker("V"), corpus.by_ticker("MA")
result = query_financials(registry, backend, corpus, concept_name="total_revenue",
                          companies=(visa, mastercard), period_year=2024)

display(pd.DataFrame([
    {"ticker": v.ticker, "tag": v.tag, "start": v.start, "end": v.end, "usd_bn": v.val / 1e9,
     "basis": v.basis, "restated": v.restated}
    for v in result.values
]))
for warning in result.warnings:
    print("WARNING:", warning, "\n")

for c in corpus:
    if c.ticker in ("AAPL", "V", "MA", "SYF"):
        print(f"{c.ticker:5} fiscal year ending in 2024: {backend.fiscal_year_end(c.cik, 2024)}")

## 12 · The coverage matrix — the deliverable

Every concept resolved across the corpus, one query per concept. Gaps are **named**, not
hidden: JPMorgan and American Express report some concepts only under custom extension
tags, which companyfacts excludes — no candidate list can recover them.

In [ ]:
from filing_copilot.structured import build_matrix

matrix = build_matrix(registry, backend, corpus, years=(2022, 2025))
pd.DataFrame([
    {
        "concept": row.concept.name,
        "scope": row.concept.applies_to,
        "resolved": f"{len(row.resolved_ciks)}/{len(row.expected_ciks)}",
        "tags used": len(row.distinct_tags),
        "fallbacks": ", ".join(by_cik[c] for c in row.fallback_ciks) or "-",
        "missing": ", ".join(by_cik[c] for c in row.missing_ciks) or "-",
    }
    for row in matrix.rows
])

## What Stage 1 outputs

| Output | Where | Used by |
|---|---|---|
| The fact table — one row per observation, partitioned `cik=/period_year=` | `data/processed/facts/` (~10 MiB) | every structured answer |
| The concept registry — business concepts → ordered tag allow-lists | `structured/concepts.yaml` | `query_financials`, coverage |
| `query_financials` — a number **plus** its tag, basis, restatement flag and warnings | `structured/financials.py` | Stage 5's MCP tool of the same name |

The number is never delivered alone. Everything needed to judge whether to trust it
travels with it.

In [ ]:
backend.close()
con.close()